# 📊 CEM4644 · MP6A — Tables
## Homework (individual): *Building shapes and their heating load*

**No coding needed.** Each grey box is one step: click ▶, wait, read the result, answer the report question. Run from top to bottom.

Photos and drawings were the last four labs. Most construction data is neither: it is a **table** (one row per mix, per building, per bid) or a **time series** (one value per hour, per day; that is MP6B). This part does the table: predict a number, predict a class, see what the model learned, then give the same job to a chat model. Every answer is checked against what really happened. About 75 minutes. No GPU needed.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line.
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp6_tabular_timeseries", "aec_tab"
FOLDERS = ["mp6_tabular_timeseries"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_tab import lab
lab.setup(dataset="homework", part="tabular")


## Part 1 · A table

768 simulated residential buildings of the same volume but different shapes, glazing and orientation, with the heating load a building-energy simulator computed for each. Every row is one building; the last column, **heating load (kWh/m²)**, is the answer the model will learn to predict from the others.

In [ ]:
#@title ▶ Step 1a · Look at the table { display-mode: "form" }
rows = 10 #@param [5, 10, 20] {type:"raw"}
lab.show_table(rows)


## Part 2 · Two questions, one table

The same table can answer **which class?** (a category: *classification*, the question you just answered yourself) or **how much?** (a number: *regression*). Both models train on 80 % of the rows and are scored on the 20 % they never saw.

In [ ]:
#@title ▶ Step 2a · Classification: predict the class { display-mode: "form" }
#@markdown *grades* puts each building in one of 4 bands of heating load, the game of Step 1b; *pass / fail* asks whether it reaches the *threshold* you set.
model = "decision trees (gradient boosting)" #@param ["a straight line (linear regression)", "decision trees (gradient boosting)"]
task = "grades" #@param ["grades", "pass / fail against a specification"]
threshold = 20 #@param {type:"slider", min:8, max:40, step:2}
lab.classification(model, task, threshold)


In [ ]:
#@title ▶ Step 2b · Regression: predict the number { display-mode: "form" }
model = "decision trees (gradient boosting)" #@param ["a straight line (linear regression)", "decision trees (gradient boosting)"]
lab.regression(model)


> ### 📝 Report question 1
> From Step 2a and 2b: the average miss and the share of buildings in the right band. These numbers are far better than the concrete table's in the workshop. What is different about this table (read the description in Part 1), and why does that make it easier for a model? Would you trust a model trained on it for a real building?

## Part 3 · What the model learned

A model that scores well may still have learned the wrong thing. Two checks: which columns it leans on, and how its prediction moves when you change one input at a time.

In [ ]:
#@title ▶ Step 3a · Which columns matter { display-mode: "form" }
lab.importance()


In [ ]:
#@title ▶ Step 3b · What if… { display-mode: "form" }
#@markdown Move a slider; the prediction updates. Everything not on a slider stays as it is in the chosen row.
start_from = "a typical row" #@param ["a typical row", "row 12", "row 100", "row 500"]
lab.whatif(start_from)


> ### 📝 Report question 2
> From Step 3: which two columns decide the heating load, and in which direction? Set the sliders to a building you would design yourself and report its predicted load. Does the model tell you anything a building-energy simulator would not?

## Part 4 · The same job, by a chat model

**hokie.ai** (https://hokie.ai.vt.edu/, Virginia Tech's free access to GPT models, sign in with your VT account) gets the same training buildings the models above learned from, and 30 of the held-out buildings without their heating load. You paste its reply back into the notebook, which scores it against what really happened, next to the notebook's own models. First the chat on its own, then the chat told to use its **data-analysis tool** (it writes and runs code on the files). The 30 buildings are the same for everyone, so you can compare with your neighbours.

In [ ]:
#@title ▶ Step 4a · Ask the chat { display-mode: "form" }
#@markdown Run the same prompt in **two** new chats and score both replies: the table then compares them. If attaching files does not work, choose *paste the data into the prompt* and run the cell again.
give = "attach the files" #@param ["attach the files", "paste the data into the prompt"]
lab.chat_table(give)


In [ ]:
#@title ▶ Step 4b · Ask the chat to use its analysis tool { display-mode: "form" }
#@markdown Pick the model the chat should train, run the cell, and follow the steps. If the reply shows no code or analysis panel, ask it again to *use your data-analysis tool*. Then try a second model: every reply you score stays in the table.
model = "gradient-boosted trees" #@param ["gradient-boosted trees", "a random forest", "a straight line", "a small neural network"]
lab.chat_table_tool(model)


> ### 📝 Report question 3
> From Step 4a and 4b: the chat's average miss on its own and with its analysis tool (two models), against the notebook's trees. This table comes from a simulator and the trees nearly get it perfect (question 1): did the chat on its own come close? What does that tell you about the difference between reasoning about a table and fitting a model to it?

## Part 5 · Your own table

A small app, opened from a link: upload any CSV, pick the column to predict, and it fits decision trees and scores them on held-out rows.

### 🔨 The tap test: data you collect yourself

Knock on a wall and you can hear whether a stud is behind it; inspectors do the same on concrete and tile to find hollow, delaminated spots (a *sounding test*). Here you record taps with your phone, turn each tap into one row of a table, and train a model to tell the surfaces apart.

1. **Plan.** Pick 4–5 surfaces (for example drywall between studs, drywall over a stud, concrete or masonry, a wooden door, glass or tile) and 3–4 separate spots of each. To be sure a spot is over a stud, use the magnetometer in the free *phyphox* app: it jumps at the drywall screws.
2. **Record.** At each spot, tap 10 times, about a second apart, always with the same object (a coin or a pen cap). Hold the phone about 20 cm away in a quiet room and use its voice recorder: one recording per spot, named by what it is (`hollow_bedroom_1.m4a`). About 200 taps in all.
3. **Make the table.** Upload the recordings to the notebook. It finds each tap and measures its pitch, brightness, ring time, loudness and low / mid / high energy: one row per tap, with the spot and the surface. Download the table: it is your dataset.
4. **Look first.** Write down what you expect (does hollow sound lower? ring longer?), then check it against each surface's average spectrum and a scatter plot of two measurements.
5. **Train and test twice:** once with the taps split at random, once with whole spots held out. Why do the two scores differ, and which would you believe?
6. **Blind test.** Tap a spot you did not record, let the model name the surface, and check.
7. **Ask the chat.** Give hokie.ai your table, ask it to do the same with its analysis tool, and compare.

Report which surfaces get confused, which measurements matter, and whether the model would still work in another room, with another phone or with another person tapping.

In [ ]:
#@title ▶ Step 5 · Your own table { display-mode: "form" }
#@markdown Open the printed link in a new tab.
lab.upload_app()


> ### 📝 Report question 4
> The main deliverable: find or make a table of your own (a bid tabulation, a materials price list, anything with a numeric column to predict and 30+ rows). Run it through Step 5 and report what the data is, what the app found, and what you would need to trust the numbers. Then give the same file to the chat and ask it to use its analysis tool to do the same: does it agree with the app?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
lab.report_summary()


### Credits
- Table: Energy Efficiency, A. Tsanas and A. Xifara (2012), UCI Machine Learning Repository, CC BY 4.0, https://archive.ics.uci.edu/dataset/242/energy+efficiency.
- Chat model: the GPT models behind hokie.ai (Virginia Tech).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp6_tabular_timeseries`).